# State Create


In [1]:
from typing import Annotated, Sequence
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

In [2]:
# this is the memory or state of our agent
class AgentState(TypedDict):
    #the add_message function ensures that new messages added to old messages, not deleted.
    messages: Annotated[Sequence[BaseMessage], add_messages]

# Setting up model and tools

i will create a simple calculator tools and connect it to the OpenAI model

In [3]:
import os
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

creating a simple tool

In [4]:
#command to load the automatic key from .env file
load_dotenv()

@tool
def multiply (a:int, b: int) -> int:
    """use the tool to multiply two numbers"""
    return a * b

# putting the tool on the list
tools = [multiply]

# OpenAI model setup and connecting to tools
model = ChatOpenAI(model="gpt-4o", temperature=0)
model_with_tools = model.bind_tools(tools)

# Creating Nodes

In [5]:
# we will  create two nodes: one for the model and the other for running the tool
from langgraph.prebuilt import ToolNode

# model node: which will call the model with the users message
def call_model(state:AgentState):
    messages = state['messages']
    respons = model_with_tools.invoke(messages)
    return {'messages':[respons]}

# tool node: Tools can be easily run using LangGraph pre build
tool_node = ToolNode(tools)

# Routing or Decision making logic

In [6]:
from langgraph.graph import END

def should_continue(state:AgentState):
    messages = state['messages']
    last_message = messages[-1]
    
    # if the model ask to call a tool
    if last_message.tool_calls:
        return 'tools'
    
    #if there are no tool calls, this is end
    return 'end'

# Building The Graph

In [9]:
from langgraph.graph import StateGraph

#initializing the graph structure
workflow = StateGraph(AgentState)

#adding nodes to the graph
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

#setting the starting point node
workflow.set_entry_point("agent")

#establishing connections between nodes
workflow.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools":"tools",
        "end": END
    }
)

#the edge of returning to the agent after using the tool
workflow.add_edge("tools", "agent")

#Compile the graph to run
app = workflow.compile()